## Cài đặt và Import thư viện

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 1-VLSP2018-SA-Restaurant-train (7-3-2018).txt to 1-VLSP2018-SA-Restaurant-train (7-3-2018).txt
Saving 2-VLSP2018-SA-Restaurant-dev (7-3-2018).txt to 2-VLSP2018-SA-Restaurant-dev (7-3-2018).txt
Saving 3-VLSP2018-SA-Restaurant-test (8-3-2018).txt to 3-VLSP2018-SA-Restaurant-test (8-3-2018).txt


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from gensim.models import KeyedVectors

In [ ]:
TRAIN_PATH = "1-VLSP2018-SA-Restaurant-train (7-3-2018).txt"
DEV_PATH   = "2-VLSP2018-SA-Restaurant-dev (7-3-2018).txt"
TEST_PATH  = "3-VLSP2018-SA-Restaurant-test (8-3-2018).txt"

# ---- Embedding ----
# Nếu dùng word2vec/fasttext .vec => binary=False
# Nếu dùng word2vec .bin => binary=True
EMBEDDING_PATH = "your_embeddings.vec"
EMBEDDING_BINARY = False

# ---- Hyperparameters ----
BATCH_SIZE = 32
EPOCHS = 30
THRESHOLD = 0.5

## Tiền xử lý Dữ liệu và Nhãn (Labels)
* Hàm load_vlsp_restaurant_file giúp đọc file text của VLSP.
* Hàm parse_label_line và normalize_one_label làm nhiệm vụ trích xuất các nhãn từ định dạng thô (VD: {FOOD#QUALITY, positive}) thành một chuỗi duy nhất
* Hàm clean_text giúp làm sạch văn bản, loại bỏ các ký tự đặc biệt, chỉ giữ lại số, chữ cái và tiếng Việt.


In [ ]:
def parse_label_line(label_line):
    """
    Parse dòng kiểu:
    {FOOD#STYLE&OPTIONS, neutral}, {FOOD#QUALITY, neutral}

    Trả về:
    ['FOOD#STYLE&OPTIONS#neutral', 'FOOD#QUALITY#neutral']
    """
    results = []

    # tìm tất cả cụm {ASPECT, polarity}
    matches = re.findall(r'\{([^,{}]+)\s*,\s*(positive|negative|neutral)\}', label_line, flags=re.IGNORECASE)

    for aspect, polarity in matches:
        aspect = aspect.strip().upper()
        polarity = polarity.strip().lower()
        results.append(f"{aspect}#{polarity}")

    return results


def load_vlsp_restaurant_file(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        lines = [line.rstrip("\n") for line in f]

    samples = []
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        if not line:
            i += 1
            continue

        # bắt đầu 1 sample
        if line.startswith("#"):
            sample_id = line
            i += 1

            # bỏ dòng trống nếu có
            while i < n and not lines[i].strip():
                i += 1

            if i >= n:
                break

            # dòng text
            text = lines[i].strip()
            i += 1

            # bỏ dòng trống nếu có
            while i < n and not lines[i].strip():
                i += 1

            label_list = []

            # dòng label thường là ngay sau text, trước dòng trống hoặc # tiếp theo
            if i < n and not lines[i].strip().startswith("#"):
                label_line = lines[i].strip()
                label_list = parse_label_line(label_line)
                i += 1

            samples.append({
                "id": sample_id,
                "text": text,
                "label_list": label_list
            })

        else:
            i += 1

    return pd.DataFrame(samples)

## Mã hóa Nhãn (Label Encoding)

In [ ]:
def normalize_one_label(label_line):
    """
    Chuyển 1 dòng label về dạng:
    ENTITY#ATTRIBUTE#polarity

    Hỗ trợ cả:
    - FOOD#QUALITY positive
    - FOOD#QUALITY: positive
    - FOOD#QUALITY\tpositive
    """
    s = label_line.strip()

    # đổi ":" thành space để dễ tách
    s = s.replace(":", " ")
    s = re.sub(r"\s+", " ", s).strip()

    parts = s.split(" ")

    if len(parts) < 2:
        return None

    polarity = parts[-1].lower()
    aspect = " ".join(parts[:-1]).strip()

    # bỏ space dư
    aspect = aspect.replace(" ", "")

    # chuẩn hóa polarity
    if polarity not in {"positive", "negative", "neutral"}:
        return None

    return f"{aspect}#{polarity}"


def normalize_label_list(raw_labels):
    results = []
    for line in raw_labels:
        norm = normalize_one_label(line)
        if norm is not None:
            results.append(norm)
    return results


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = text.replace("_", " ")

    # giữ lại chữ, số, khoảng trắng và tiếng Việt có dấu
    text = re.sub(
        r"[^0-9a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ\s]",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return clean_text(text).split()

In [ ]:
train_df = load_vlsp_restaurant_file(TRAIN_PATH)
dev_df   = load_vlsp_restaurant_file(DEV_PATH)
test_df  = load_vlsp_restaurant_file(TEST_PATH)

print("Train size:", len(train_df))
print("Dev size  :", len(dev_df))
print("Test size :", len(test_df))

print("\nVí dụ train:")
print(train_df[["id", "text", "label_list"]].head(10))

Train size: 2961
Dev size  : 1290
Test size : 500

Ví dụ train:
    id                                               text  \
0   #1  _ Ảnh chụp từ hôm qua, đi chơi với gia đình và...   
1   #2  _Hương vị thơm ngon, ăn cay cay rất thích, nêm...   
2   #3  - 1 bàn tiệc hoành tráng 3 đứa ăn no muốn tắt ...   
3   #4  - Các bạn nhìn cái chảo này có to không 🙄🙄🙄- T...   
4   #5  - Cháo: có nhiều hương cho các bạn chọn, nhưng...   
5   #6  - Đồ nướng thì chỗ này không ít bạn "chẻ" biết...   
6   #7  - Đói không ngủ được nên up hình đồ ăn cho mấy...   
7   #8  - Khẩu vị vừa ăn hợp vệ sinh , không gian quán...   
8   #9  - Kimchi Kimchi 😹😹- Nói cho cùng thì đồ ăn càn...   
9  #10  - Lẩu hàu thì chưa thấy đâu có. Mới ăn lần đầu...   

                                          label_list  
0  [FOOD#STYLE&OPTIONS#neutral, FOOD#QUALITY#neut...  
1  [FOOD#QUALITY#positive, FOOD#STYLE&OPTIONS#pos...  
2  [FOOD#STYLE&OPTIONS#positive, FOOD#PRICES#posi...  
3  [FOOD#STYLE&OPTIONS#positive, RESTAURANT#

Do một câu đánh giá (review) có thể chứa nhiều nhãn cùng lúc (Multi-label), code sử dụng MultiLabelBinarizer của sklearn để chuyển danh sách các nhãn thành các vector dạng One-Hot Encoding đa lớp. Tổng cộng bài toán có 36 nhãn khía cạnh - cảm xúc khác nhau.

In [ ]:
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df["label_list"])
y_dev   = mlb.transform(dev_df["label_list"])
y_test  = mlb.transform(test_df["label_list"])

print("\nSố nhãn:", len(mlb.classes_))
print("Một vài nhãn:", mlb.classes_[:10])


Số nhãn: 36
Một vài nhãn: ['AMBIENCE#GENERAL#negative' 'AMBIENCE#GENERAL#neutral'
 'AMBIENCE#GENERAL#positive' 'DRINKS#PRICES#negative'
 'DRINKS#PRICES#neutral' 'DRINKS#PRICES#positive'
 'DRINKS#QUALITY#negative' 'DRINKS#QUALITY#neutral'
 'DRINKS#QUALITY#positive' 'DRINKS#STYLE&OPTIONS#negative']


In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.vec.gz
!gunzip cc.vi.300.vec.gz

--2026-03-21 09:03:45--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.84.118.60, 99.84.118.117, 99.84.118.67, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.84.118.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1235219084 (1.1G) [binary/octet-stream]
Saving to: ‘cc.vi.300.vec.gz’

cc.vi.300.vec.gz    100%[===================>]   1.15G  42.0MB/s    in 28s     

2026-03-21 09:04:14 (42.2 MB/s) - ‘cc.vi.300.vec.gz’ saved [1235219084/1235219084]



## Khởi tạo Word Embedding với FastText
Tự động tải bộ Pre-trained Word Embedding tiếng Việt của FastText (cc.vi.300.vec - vector 300 chiều). Sau đó dùng KeyedVectors của gensim để load các vector từ này vào bộ nhớ.

In [ ]:
EMBEDDING_PATH = "cc.vi.300.vec"
EMBEDDING_BINARY = False

embedding_model = KeyedVectors.load_word2vec_format(EMBEDDING_PATH, binary=BINARY)
embedding_dim = embedding_model.vector_size

print("Embedding dim =", embedding_dim)


Embedding dim = 300


In [ ]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
dev_df["clean_text"]   = dev_df["text"].apply(clean_text)
test_df["clean_text"]  = test_df["text"].apply(clean_text)

train_df["tokens"] = train_df["clean_text"].apply(tokenize)
dev_df["tokens"]   = dev_df["clean_text"].apply(tokenize)
test_df["tokens"]  = test_df["clean_text"].apply(tokenize)

print(train_df[["text", "clean_text", "tokens"]].head())

                                                text  \
0  _ Ảnh chụp từ hôm qua, đi chơi với gia đình và...   
1  _Hương vị thơm ngon, ăn cay cay rất thích, nêm...   
2  - 1 bàn tiệc hoành tráng 3 đứa ăn no muốn tắt ...   
3  - Các bạn nhìn cái chảo này có to không 🙄🙄🙄- T...   
4  - Cháo: có nhiều hương cho các bạn chọn, nhưng...   

                                          clean_text  \
0  ảnh chụp từ hôm qua đi chơi với gia đình và 1 ...   
1  hương vị thơm ngon ăn cay cay rất thích nêm nế...   
2  1 bàn tiệc hoành tráng 3 đứa ăn no muốn tắt th...   
3  các bạn nhìn cái chảo này có to không trên hìn...   
4  cháo có nhiều hương cho các bạn chọn nhưng mìn...   

                                              tokens  
0  [ảnh, chụp, từ, hôm, qua, đi, chơi, với, gia, ...  
1  [hương, vị, thơm, ngon, ăn, cay, cay, rất, thí...  
2  [1, bàn, tiệc, hoành, tráng, 3, đứa, ăn, no, m...  
3  [các, bạn, nhìn, cái, chảo, này, có, to, không...  
4  [cháo, có, nhiều, hương, cho, các, bạn, chọn, ..

## Mean Pooling
Hàm sentence_to_vec thực hiện việc chuyển một câu thành một vector đại diện duy nhất bằng cách tính trung bình cộng (mean) của tất cả các vector từ (tokens) xuất hiện trong câu đó.

In [ ]:
def sentence_to_vec(tokens, emb_model, emb_dim):
    vecs = []

    for tok in tokens:
        if tok in emb_model:
            vecs.append(emb_model[tok])

    if len(vecs) == 0:
        return np.zeros(emb_dim, dtype=np.float32)

    return np.mean(vecs, axis=0).astype(np.float32)


def build_X(df, emb_model, emb_dim):
    X = np.vstack([
        sentence_to_vec(tokens, emb_model, emb_dim)
        for tokens in df["tokens"]
    ])
    return X.astype(np.float32)


X_train = build_X(train_df, embedding_model, embedding_dim)
X_dev   = build_X(dev_df, embedding_model, embedding_dim)
X_test  = build_X(test_df, embedding_model, embedding_dim)

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_dev  :", X_dev.shape)
print("y_dev  :", y_dev.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)



X_train: (2961, 300)
y_train: (2961, 36)
X_dev  : (1290, 300)
y_dev  : (1290, 36)
X_test : (500, 300)
y_test : (500, 36)


## Xây dựng và Huấn luyện Mô hình FFNN
Khởi tạo mạng neuron Sequential gồm các lớp Dense (256 và 128 nơ-ron) với hàm kích hoạt relu. Vì đây là bài toán phân loại đa nhãn (Multi-label Classification), lớp output sử dụng hàm kích hoạt sigmoid (thay vì softmax) và hàm loss là binary_crossentropy để dự đoán xác suất độc lập cho từng nhãn.

In [ ]:
model = Sequential([
    Input(shape=(embedding_dim,)),
    Dense(256, activation="relu"),
    Dense(256, activation="relu"),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dense(len(mlb.classes_), activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="binary_acc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 256)            │        77,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 246,180 (961.64 KB)

 Trainable params: 246,180 (961.64 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_dev, y_dev),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
93/93 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - binary_acc: 0.9039 - loss: 0.2671 - precision: 0.4386 - recall: 0.3626 - val_binary_acc: 0.9424 - val_loss: 0.1726 - val_precision: 0.7977 - val_recall: 0.2989
Epoch 2/30
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - binary_acc: 0.9298 - loss: 0.1916 - precision: 0.7022 - recall: 0.3384 - val_binary_acc: 0.9376 - val_loss: 0.1733 - val_precision: 0.6187 - val_recall: 0.4133
Epoch 3/30
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - binary_acc: 0.9328 - loss: 0.1853 - precision: 0.7145 - recall: 0.3826 - val_binary_acc: 0.9409 - val_loss: 0.1663 - val_precision: 0.6659 - val_recall: 0.4063
Epoch 4/30
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - binary_acc: 0.9345 - loss: 0.1794 - precision: 0.7308 - recall: 0.3945 - val_binary_acc: 0.9411 - val_loss: 0.1624 - val_precision: 0.6493 - val_recall: 0.4464
Epoch 5/30
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - binary_acc: 0.9362 - loss: 0.1718 - precision: 0.7321 - recall: 0.4239 - val_binary_acc: 0.9413 -

In [ ]:
def evaluate_model(model, X, y_true, threshold=0.5, split_name="TEST"):
    y_prob = model.predict(X, verbose=0)
    y_pred = (y_prob >= threshold).astype(int)

    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    samples_f1 = f1_score(y_true, y_pred, average="samples", zero_division=0)

    print(f"\n===== {split_name} EVALUATION =====")
    print("Micro F1  :", micro_f1)
    print("Macro F1  :", macro_f1)
    print("Samples F1:", samples_f1)

    print(f"\n===== {split_name} Classification Report =====")
    print(classification_report(
        y_true,
        y_pred,
        target_names=mlb.classes_,
        zero_division=0
    ))

    return y_prob, y_pred


dev_prob, dev_pred = evaluate_model(model, X_dev, y_dev, threshold=THRESHOLD, split_name="DEV")
test_prob, test_pred = evaluate_model(model, X_test, y_test, threshold=THRESHOLD, split_name="TEST")


===== DEV EVALUATION =====
Micro F1  : 0.5796274542708508
Macro F1  : 0.10104877031174134
Samples F1: 0.5744961240310077

===== DEV Classification Report =====
                                   precision    recall  f1-score   support

        AMBIENCE#GENERAL#negative       0.00      0.00      0.00        33
         AMBIENCE#GENERAL#neutral       0.00      0.00      0.00        23
        AMBIENCE#GENERAL#positive       0.59      0.28      0.37       149
           DRINKS#PRICES#negative       0.00      0.00      0.00         2
            DRINKS#PRICES#neutral       0.00      0.00      0.00        13
           DRINKS#PRICES#positive       0.00      0.00      0.00        29
          DRINKS#QUALITY#negative       0.00      0.00      0.00         3
           DRINKS#QUALITY#neutral       0.00      0.00      0.00         6
          DRINKS#QUALITY#positive       0.00      0.00      0.00        30
    DRINKS#STYLE&OPTIONS#negative       0.00      0.00      0.00         3
     DRINKS#S

In [ ]:
def evaluate_model(model, X, y_true, threshold=0.5, split_name="TEST"):
    y_prob = model.predict(X, verbose=0)
    y_pred = (y_prob >= threshold).astype(int)

    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    samples_f1 = f1_score(y_true, y_pred, average="samples", zero_division=0)

    print(f"\n===== {split_name} EVALUATION =====")
    print("Micro F1  :", micro_f1)
    print("Macro F1  :", macro_f1)
    print("Samples F1:", samples_f1)

    print(f"\n===== {split_name} Classification Report =====")
    print(classification_report(
        y_true,
        y_pred,
        target_names=mlb.classes_,
        zero_division=0
    ))

    return y_prob, y_pred


dev_prob, dev_pred = evaluate_model(model, X_dev, y_dev, threshold=THRESHOLD, split_name="DEV")
test_prob, test_pred = evaluate_model(model, X_test, y_test, threshold=THRESHOLD, split_name="TEST")


===== DEV EVALUATION =====
Micro F1  : 0.5762945123000172
Macro F1  : 0.10311924493290911
Samples F1: 0.5685418973791067

===== DEV Classification Report =====
                                   precision    recall  f1-score   support

        AMBIENCE#GENERAL#negative       0.00      0.00      0.00        33
         AMBIENCE#GENERAL#neutral       0.00      0.00      0.00        23
        AMBIENCE#GENERAL#positive       0.64      0.34      0.45       149
           DRINKS#PRICES#negative       0.00      0.00      0.00         2
            DRINKS#PRICES#neutral       0.00      0.00      0.00        13
           DRINKS#PRICES#positive       0.00      0.00      0.00        29
          DRINKS#QUALITY#negative       0.00      0.00      0.00         3
           DRINKS#QUALITY#neutral       0.00      0.00      0.00         6
          DRINKS#QUALITY#positive       0.00      0.00      0.00        30
    DRINKS#STYLE&OPTIONS#negative       0.00      0.00      0.00         3
     DRINKS#S

In [ ]:
def predict_review(review_text, model, mlb, emb_model, emb_dim, threshold=0.5):
    tokens = tokenize(review_text)
    x = sentence_to_vec(tokens, emb_model, emb_dim).reshape(1, -1)
    prob = model.predict(x, verbose=0)[0]

    labels = []
    for label, p in zip(mlb.classes_, prob):
        if p >= threshold:
            labels.append((label, float(p)))

    labels.sort(key=lambda x: x[1], reverse=True)
    return labels

In [ ]:
sample_review = "Nhà hàng đẹp, đồ ăn ngon nhưng phục vụ hơi chậm."
preds = predict_review(
    sample_review,
    model,
    mlb,
    embedding_model,
    embedding_dim,
    threshold=0.4
)

print("\n===== REVIEW MỚI =====")
print(sample_review)
for label, score in preds:
    print(f"{label}: {score:.4f}")



===== REVIEW MỚI =====
Nhà hàng đẹp, đồ ăn ngon nhưng phục vụ hơi chậm.
SERVICE#GENERAL#positive: 0.9516
FOOD#QUALITY#positive: 0.7596
AMBIENCE#GENERAL#positive: 0.4246


In [ ]:
model.save("ffnn_absa_model.keras")

pd.Series(mlb.classes_).to_csv("label_classes.csv", index=False, header=False)

print("\nĐã lưu model vào: ffnn_absa_model.keras")
print("Đã lưu label classes vào: label_classes.csv")


Đã lưu model vào: ffnn_absa_model.keras
Đã lưu label classes vào: label_classes.csv
